Cell 1 — Setup (installs, imports, config, device, reproducibility, and path checks)

In [13]:
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as e:
    raise RuntimeError(
        "PyTorch is not available. Install it first, e.g.:\n"
        "pip install torch torchvision\n"
        f"Original error: {e}"
    )

try:
    import SimpleITK as sitk
except Exception as e:
    raise RuntimeError(
        "SimpleITK is not available. Install it first, e.g.:\n"
        "pip install SimpleITK\n"
        f"Original error: {e}"
    )

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class Config:
    data_root: Path = Path("../../notebooks/data/rsna_mgmt")
    train_dir: str = "train"
    labels_csv: str = "train_labels.csv"
    cache_dir: str = "cache_128"
    target_shape: Tuple[int, int, int] = (128, 128, 128)
    seed: int = 42
    val_size: float = 0.2
    batch_size: int = 1
    num_workers: int = 0
    epochs: int = 8
    lr: float = 2e-4
    weight_decay: float = 1e-3
    grad_clip_norm: float = 2.0
    ema_decay: float = 0.999
    ckpt_path: Path = Path("best.pt")


def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_all(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


cfg = Config(data_root=Path("data/rsna_mgmt"))
device = resolve_device()
seed_all(cfg.seed)

data_root = cfg.data_root.expanduser().resolve()
train_root = (data_root / cfg.train_dir).resolve()
labels_path = (data_root / cfg.labels_csv).resolve()
cache_root = (data_root / cfg.cache_dir).resolve()
cache_root.mkdir(parents=True, exist_ok=True)

status = {
    "device": str(device),
    "data_root": str(data_root),
    "train_root_exists": train_root.exists(),
    "labels_csv_exists": labels_path.exists(),
    "cache_root": str(cache_root),
}
print(status)

if not train_root.exists():
    raise FileNotFoundError(
        f"Training folder not found: {train_root}\n"
        "Expected structure:\n"
        "  data/rsna_mgmt/\n"
        "    train/\n"
        "    train_labels.csv\n"
        "Put the Kaggle dataset there or change cfg.data_root."
    )

if not labels_path.exists():
    raise FileNotFoundError(
        f"Labels CSV not found: {labels_path}\n"
        "Expected file: train_labels.csv in your data_root."
    )

{'device': 'mps', 'data_root': '/Users/pranav/Documents/GitHub/TumorFinder/notebooks/data/rsna_mgmt', 'train_root_exists': False, 'labels_csv_exists': False, 'cache_root': '/Users/pranav/Documents/GitHub/TumorFinder/notebooks/data/rsna_mgmt/cache_128'}


FileNotFoundError: Training folder not found: /Users/pranav/Documents/GitHub/TumorFinder/notebooks/data/rsna_mgmt/train
Expected structure:
  data/rsna_mgmt/
    train/
    train_labels.csv
Put the Kaggle dataset there or change cfg.data_root.

Cell 2 — Build the patient table (ID → label + where the 4 MRI modalities live)

In [ ]:
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

labels = pd.read_csv(labels_path)

required = {"BraTS21ID", "MGMT_value"}
if not required.issubset(labels.columns):
    raise ValueError(f"Expected columns {required}, got {set(labels.columns)}")

labels = labels.copy()
labels["BraTS21ID"] = labels["BraTS21ID"].astype(str).str.zfill(5)
labels["MGMT_value"] = labels["MGMT_value"].astype(int)

print({"labels_rows": len(labels), "label_counts": labels["MGMT_value"].value_counts().to_dict()})


def _clean_name(s: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "", s.upper())


def _infer_modality(dir_name: str) -> Optional[str]:
    u = _clean_name(dir_name)

    if "FLAIR" in u:
        return "FLAIR"

    if "T1CE" in u or ("T1" in u and "CE" in u) or "POST" in u or "CONTRAST" in u:
        return "T1wCE"

    if "T2" in u:
        return "T2w"

    if "T1" in u:
        return "T1w"

    return None


def _list_dirs(p: Path) -> List[Path]:
    return sorted([x for x in p.iterdir() if x.is_dir()])


def build_index(train_root: Path, labels_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int], Counter]:
    rows: List[Dict[str, object]] = []
    missing = {"missing_patient_dir": 0, "missing_modalities": 0}
    seen_dirnames: Counter = Counter()

    for pid, y in tqdm(list(zip(labels_df["BraTS21ID"], labels_df["MGMT_value"])), desc="index"):
        pdir = train_root / pid
        if not pdir.exists():
            missing["missing_patient_dir"] += 1
            continue

        mods: Dict[str, Path] = {}
        for d in _list_dirs(pdir):
            seen_dirnames[d.name] += 1
            m = _infer_modality(d.name)
            if m is None:
                continue
            if m in mods:
                keep = mods[m]
                a = _clean_name(d.name)
                b = _clean_name(keep.name)
                if ("CE" in a) and ("CE" not in b):
                    mods[m] = d
                continue
            mods[m] = d

        needed = ["T1w", "T1wCE", "T2w", "FLAIR"]
        if not all(k in mods for k in needed):
            missing["missing_modalities"] += 1
            continue

        rows.append(
            {
                "BraTS21ID": pid,
                "y": int(y),
                "T1w": str(mods["T1w"]),
                "T1wCE": str(mods["T1wCE"]),
                "T2w": str(mods["T2w"]),
                "FLAIR": str(mods["FLAIR"]),
            }
        )

    return pd.DataFrame(rows), missing, seen_dirnames


df, missing, dir_counts = build_index(train_root, labels)
print({"usable_patients": len(df), "missing": missing})
display(df.head(5))

top_dirnames = dir_counts.most_common(15)
print("Top modality folder names seen (for sanity-check):")
for name, c in top_dirnames:
    print(f"{c:6d}  {name}")

if len(df) == 0:
    raise RuntimeError(
        "No usable patients found. This usually means modality folder names don't match the inference rules.\n"
        "Look at the printed folder names above and we will adjust the modality inference to your exact structure."
    )

Cell 3 — Load one MRI modality from DICOM into a real 3D volume (the “make MRI usable” step)

In [ ]:
from typing import Sequence

def _select_series_id(series_dir: str, series_ids: Sequence[str]) -> str:
    reader = sitk.ImageSeriesReader()
    best_sid = series_ids[0]
    best_n = -1
    best_key = (-1, -1, -1)

    for sid in series_ids:
        files = reader.GetGDCMSeriesFileNames(series_dir, sid)
        n = len(files)
        if n == 0:
            continue

        r = sitk.ImageFileReader()
        r.SetFileName(files[0])
        r.ReadImageInformation()
        sx = int(r.GetSize()[0])
        sy = int(r.GetSize()[1])
        key = (n, sx, sy)

        if key > best_key:
            best_key = key
            best_n = n
            best_sid = sid

    if best_n < 1:
        raise FileNotFoundError(f"No readable DICOM slices in: {series_dir}")

    return best_sid


def load_dicom_volume(series_dir: str) -> np.ndarray:
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(series_dir)
    if not series_ids:
        raise FileNotFoundError(f"No DICOM series found in: {series_dir}")

    sid = _select_series_id(series_dir, series_ids)
    files = reader.GetGDCMSeriesFileNames(series_dir, sid)
    if len(files) < 8:
        raise ValueError(f"Too few DICOM slices (n={len(files)}) in: {series_dir}")

    reader.SetFileNames(files)
    img = reader.Execute()
    vol = sitk.GetArrayFromImage(img).astype(np.float32)

    if vol.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {vol.shape} from {series_dir}")

    if not np.isfinite(vol).all():
        raise ValueError(f"Non-finite values detected in volume from {series_dir}")

    return vol


row0 = df.iloc[0]
pid0 = row0["BraTS21ID"]

paths = {
    "T1w": str(row0["T1w"]),
    "T1wCE": str(row0["T1wCE"]),
    "T2w": str(row0["T2w"]),
    "FLAIR": str(row0["FLAIR"]),
}

vols = {k: load_dicom_volume(v) for k, v in paths.items()}

summary = {}
for k, v in vols.items():
    summary[k] = {
        "shape": tuple(int(x) for x in v.shape),
        "min": float(np.min(v)),
        "max": float(np.max(v)),
        "mean": float(np.mean(v)),
        "std": float(np.std(v)),
    }

print({"patient": pid0, **summary})

d_mid = {k: v.shape[0] // 2 for k, v in vols.items()}
plt.figure(figsize=(10, 3))
plt.subplot(1, 4, 1); plt.imshow(vols["T1w"][d_mid["T1w"]], cmap="gray"); plt.axis("off"); plt.title("T1w")
plt.subplot(1, 4, 2); plt.imshow(vols["T1wCE"][d_mid["T1wCE"]], cmap="gray"); plt.axis("off"); plt.title("T1wCE")
plt.subplot(1, 4, 3); plt.imshow(vols["T2w"][d_mid["T2w"]], cmap="gray"); plt.axis("off"); plt.title("T2w")
plt.subplot(1, 4, 4); plt.imshow(vols["FLAIR"][d_mid["FLAIR"]], cmap="gray"); plt.axis("off"); plt.title("FLAIR")
plt.tight_layout()
plt.show()

Cell 4 — Preprocess into the exact tensor your 3D model needs

In [ ]:
from typing import Dict, Tuple, List

def zscore_nonzero(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    if x.ndim != 3:
        raise ValueError(f"Expected 3D volume, got {x.ndim}D")
    m = x > 0
    if int(m.sum()) < 64:
        return x.astype(np.float32, copy=False)
    mu = float(x[m].mean())
    sd = float(x[m].std())
    return ((x - mu) / (sd + eps)).astype(np.float32, copy=False)

def center_crop_pad_3d(x: np.ndarray, target: Tuple[int, int, int]) -> np.ndarray:
    if x.ndim != 3:
        raise ValueError(f"Expected 3D volume, got {x.ndim}D")
    td, th, tw = target
    d, h, w = x.shape
    out = np.zeros((td, th, tw), dtype=x.dtype)

    sd0 = max(0, (d - td) // 2)
    sh0 = max(0, (h - th) // 2)
    sw0 = max(0, (w - tw) // 2)

    sd1 = min(d, sd0 + td)
    sh1 = min(h, sh0 + th)
    sw1 = min(w, sw0 + tw)

    dd0 = max(0, (td - d) // 2)
    dh0 = max(0, (th - h) // 2)
    dw0 = max(0, (tw - w) // 2)

    dd1 = dd0 + (sd1 - sd0)
    dh1 = dh0 + (sh1 - sh0)
    dw1 = dw0 + (sw1 - sw0)

    out[dd0:dd1, dh0:dh1, dw0:dw1] = x[sd0:sd1, sh0:sh1, sw0:sw1]
    return out

def preprocess_patient_from_row(row: pd.Series, target: Tuple[int, int, int]) -> np.ndarray:
    order = ["T1w", "T1wCE", "T2w", "FLAIR"]
    vols: List[np.ndarray] = []
    for m in order:
        v = load_dicom_volume(str(row[m]))
        v = zscore_nonzero(v)
        v = center_crop_pad_3d(v, target)
        if not np.isfinite(v).all():
            raise ValueError(f"Non-finite values after preprocessing in modality {m} for patient {row['BraTS21ID']}")
        vols.append(v.astype(np.float32, copy=False))
    x = np.stack(vols, axis=0)
    if x.shape != (4, *target):
        raise ValueError(f"Bad stacked shape: {x.shape}, expected {(4, *target)}")
    return x

x0 = preprocess_patient_from_row(df.iloc[0], cfg.target_shape)
y0 = int(df.iloc[0]["y"])
print({"patient": df.iloc[0]["BraTS21ID"], "y": y0, "x_shape": x0.shape, "dtype": str(x0.dtype), "min": float(x0.min()), "max": float(x0.max())})

mid = x0.shape[1] // 2
plt.figure(figsize=(10, 3))
plt.subplot(1, 4, 1); plt.imshow(x0[0, mid], cmap="gray"); plt.axis("off"); plt.title("T1w")
plt.subplot(1, 4, 2); plt.imshow(x0[1, mid], cmap="gray"); plt.axis("off"); plt.title("T1wCE")
plt.subplot(1, 4, 3); plt.imshow(x0[2, mid], cmap="gray"); plt.axis("off"); plt.title("T2w")
plt.subplot(1, 4, 4); plt.imshow(x0[3, mid], cmap="gray"); plt.axis("off"); plt.title("FLAIR")
plt.tight_layout()
plt.show()

Cell 5 — Cache every patient as a ready-to-train tensor (so training is fast and stable)

In [ ]:
from typing import List

def cache_path_for(pid: str, target: Tuple[int, int, int]) -> Path:
    d, h, w = target
    return cache_root / f"{pid}_{d}x{h}x{w}.npz"

def write_npz_atomic(path: Path, x: np.ndarray, y: int) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    np.savez_compressed(tmp, x=x.astype(np.float32, copy=False), y=np.int64(y))
    os.replace(tmp, path)

def cache_one(row: pd.Series, target: Tuple[int, int, int]) -> str:
    pid = str(row["BraTS21ID"])
    y = int(row["y"])
    out = cache_path_for(pid, target)
    if out.exists():
        return str(out)
    x = preprocess_patient_from_row(row, target)
    write_npz_atomic(out, x, y)
    return str(out)

cache_paths: List[str] = []
failed: List[str] = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="caching"):
    try:
        cache_paths.append(cache_one(row, cfg.target_shape))
    except Exception as e:
        failed.append(f"{row['BraTS21ID']}: {type(e).__name__}: {e}")
        cache_paths.append("")

df = df.copy()
df["cache_path"] = cache_paths

usable = df["cache_path"].astype(str).str.len() > 0
df = df[usable].reset_index(drop=True)

print({"cached_patients": int(len(df)), "failed_patients": int(len(failed))})
if failed:
    print("First 5 failures:")
    for s in failed[:5]:
        print(s)

test = np.load(df.loc[0, "cache_path"])
x_test = test["x"]
y_test = int(test["y"])
print({"cache_file": df.loc[0, "cache_path"], "x_shape": tuple(x_test.shape), "x_dtype": str(x_test.dtype), "y": y_test})

Cell 6 — Train/Validation split + DataLoaders (so the model can “eat” your cached MRI tensors)

In [ ]:
from typing import Tuple

df = df.copy()
if "cache_path" not in df.columns:
    raise ValueError("df must contain 'cache_path' from caching step")

if len(df) < 50:
    raise ValueError(f"Too few cached patients to train reliably: {len(df)}")

train_df, val_df = train_test_split(
    df,
    test_size=float(cfg.val_size),
    random_state=int(cfg.seed),
    stratify=df["y"].astype(int),
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_pos = int(train_df["y"].astype(int).sum())
train_neg = int(len(train_df) - train_pos)
pos_weight_value = float(train_neg / max(1, train_pos))
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32)

print(
    {
        "train_n": int(len(train_df)),
        "val_n": int(len(val_df)),
        "train_pos": train_pos,
        "train_neg": train_neg,
        "pos_weight": pos_weight_value,
    }
)

class MGMTCachedDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, expected_shape: Tuple[int, int, int, int]):
        self.paths = frame["cache_path"].astype(str).tolist()
        self.labels = frame["y"].astype(int).tolist()
        self.expected_shape = expected_shape

        if len(self.paths) != len(self.labels):
            raise ValueError("Mismatch between cache paths and labels length")

        bad = [p for p in self.paths if not Path(p).exists()]
        if bad:
            raise FileNotFoundError(f"Missing cache files (showing up to 3): {bad[:3]}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        p = self.paths[idx]
        d = np.load(p)
        x = d["x"]
        y = float(self.labels[idx])

        if x.shape != self.expected_shape:
            raise ValueError(f"Bad x shape in {p}: {x.shape}, expected {self.expected_shape}")
        if x.dtype != np.float32:
            x = x.astype(np.float32, copy=False)

        xt = torch.from_numpy(x)
        yt = torch.tensor(y, dtype=torch.float32)
        return xt, yt

expected = (4, *cfg.target_shape)
train_set = MGMTCachedDataset(train_df, expected_shape=expected)
val_set = MGMTCachedDataset(val_df, expected_shape=expected)

pin = bool(cfg.pin_memory and device.type == "cuda")

train_loader = DataLoader(
    train_set,
    batch_size=int(cfg.batch_size),
    shuffle=True,
    num_workers=int(cfg.num_workers),
    pin_memory=pin,
    drop_last=False,
)

val_loader = DataLoader(
    val_set,
    batch_size=int(cfg.batch_size),
    shuffle=False,
    num_workers=int(cfg.num_workers),
    pin_memory=pin,
    drop_last=False,
)

xb, yb = next(iter(train_loader))
print({"batch_x": tuple(xb.shape), "batch_y": tuple(yb.shape), "x_dtype": str(xb.dtype), "y_dtype": str(yb.dtype)})

Cell 7 — The 3D MRI model (what it does, in beginner terms)

In [ ]:
from typing import Dict, Optional

class ConvNormAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1, groups: int = 8):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        g = min(groups, out_ch)
        while g > 1 and (out_ch % g) != 0:
            g -= 1
        self.norm = nn.GroupNorm(num_groups=g, num_channels=out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(self.conv(x)))


class ResidualBlock(nn.Module):
    def __init__(self, ch: int, groups: int = 8):
        super().__init__()
        self.a = ConvNormAct(ch, ch, k=3, s=1, p=1, groups=groups)
        self.b = nn.Conv3d(ch, ch, kernel_size=3, stride=1, padding=1, bias=False)
        g = min(groups, ch)
        while g > 1 and (ch % g) != 0:
            g -= 1
        self.norm = nn.GroupNorm(num_groups=g, num_channels=ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        r = x
        x = self.a(x)
        x = self.norm(self.b(x))
        return F.silu(x + r, inplace=True)


class MGMTResNet3D(nn.Module):
    def __init__(self, in_ch: int = 4, base: int = 24):
        super().__init__()
        self.stem = ConvNormAct(in_ch, base, k=3, s=2, p=1)
        self.s1 = nn.Sequential(ResidualBlock(base), ResidualBlock(base))
        self.d1 = ConvNormAct(base, base * 2, k=3, s=2, p=1)
        self.s2 = nn.Sequential(ResidualBlock(base * 2), ResidualBlock(base * 2))
        self.d2 = ConvNormAct(base * 2, base * 4, k=3, s=2, p=1)
        self.s3 = nn.Sequential(ResidualBlock(base * 4), ResidualBlock(base * 4))
        self.d3 = ConvNormAct(base * 4, base * 6, k=3, s=2, p=1)
        self.s4 = nn.Sequential(ResidualBlock(base * 6), ResidualBlock(base * 6))
        self.head = nn.Linear(base * 6, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.s1(x)
        x = self.d1(x)
        x = self.s2(x)
        x = self.d2(x)
        x = self.s3(x)
        x = self.d3(x)
        x = self.s4(x)
        x = x.mean(dim=(2, 3, 4))
        z = self.head(x).squeeze(1)
        return z


class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = float(decay)
        self.shadow: Dict[str, torch.Tensor] = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone()

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module) -> Dict[str, torch.Tensor]:
        backup: Dict[str, torch.Tensor] = {}
        for n, p in model.named_parameters():
            if n in self.shadow:
                backup[n] = p.detach().clone()
                p.copy_(self.shadow[n])
        return backup

    @torch.no_grad()
    def restore(self, model: nn.Module, backup: Dict[str, torch.Tensor]) -> None:
        for n, p in model.named_parameters():
            if n in backup:
                p.copy_(backup[n])


model = MGMTResNet3D(in_ch=4, base=24).to(device)
ema = EMA(model, decay=cfg.ema_decay)

trainable_params = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
print({"trainable_params": trainable_params})

xb, yb = next(iter(train_loader))
xb = xb.to(device)
with torch.no_grad():
    z = model(xb)
print({"logits_shape": tuple(z.shape), "logits_dtype": str(z.dtype)})

Cell 8 — Training + Validation (this is where the model actually learns)

In [ ]:
from typing import Dict, List, Tuple

def _to_numpy(x: torch.Tensor) -> np.ndarray:
    return x.detach().float().cpu().numpy()

@torch.no_grad()
def predict_val_probs(loader: DataLoader, use_ema: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    backup = None
    if use_ema:
        backup = ema.apply_to(model)

    ys: List[float] = []
    ps: List[float] = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        z = model(x)
        p = torch.sigmoid(z)
        ys.extend(_to_numpy(y).tolist())
        ps.extend(_to_numpy(p).tolist())

    if use_ema and backup is not None:
        ema.restore(model, backup)

    y_true = np.asarray(ys, dtype=np.float32)
    p_pred = np.asarray(ps, dtype=np.float32)
    return y_true, p_pred

def compute_metrics(y_true: np.ndarray, p_pred: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
    y_true_i = y_true.astype(np.int32)
    y_hat = (p_pred >= thr).astype(np.int32)

    cm = confusion_matrix(y_true_i, y_hat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    tpr = tp / max(1, (tp + fn))
    tnr = tn / max(1, (tn + fp))

    auc = float("nan")
    if len(np.unique(y_true_i)) > 1:
        auc = float(roc_auc_score(y_true, p_pred))

    return {
        "auc": float(auc),
        "acc": float(acc),
        "tpr": float(tpr),
        "tnr": float(tnr),
        "tp": float(tp),
        "tn": float(tn),
        "fp": float(fp),
        "fn": float(fn),
    }

def save_checkpoint(path: Path, epoch: int, best_auc: float) -> None:
    payload = {
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "cfg": cfg.__dict__,
        "model_state": model.state_dict(),
        "ema_shadow": {k: v.detach().cpu() for k, v in ema.shadow.items()},
    }
    torch.save(payload, path)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg.lr), weight_decay=float(cfg.weight_decay))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(cfg.epochs))

use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_auc = -1.0
history: List[Dict[str, float]] = []

for epoch in range(1, int(cfg.epochs) + 1):
    model.train()
    running_loss = 0.0
    steps = 0

    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            z = model(x)
            loss = criterion(z, y)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss encountered: {loss.item()}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.grad_clip_norm))
        scaler.step(optimizer)
        scaler.update()

        ema.update(model)

        running_loss += float(loss.detach().cpu().item())
        steps += 1

    scheduler.step()

    train_loss = running_loss / max(1, steps)
    y_true, p_pred = predict_val_probs(val_loader, use_ema=True)
    m = compute_metrics(y_true, p_pred, thr=0.5)

    record = {"epoch": float(epoch), "train_loss": float(train_loss), **m}
    history.append(record)

    print(
        {
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_auc": None if not math.isfinite(m["auc"]) else round(m["auc"], 6),
            "val_acc": round(m["acc"], 6),
            "val_tpr": round(m["tpr"], 6),
            "val_tnr": round(m["tnr"], 6),
            "tp": int(m["tp"]),
            "tn": int(m["tn"]),
            "fp": int(m["fp"]),
            "fn": int(m["fn"]),
        }
    )

    if math.isfinite(m["auc"]) and m["auc"] > best_auc:
        best_auc = float(m["auc"])
        save_checkpoint(cfg.ckpt_path, epoch=epoch, best_auc=best_auc)

print({"best_auc": float(best_auc), "checkpoint": str(cfg.ckpt_path.resolve())})

Cell 9 — Load the best checkpoint and report final validation results (AUC, ROC curve, confusion matrix)

In [ ]:
import math
from typing import Dict, Tuple

if not cfg.ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path.resolve()}")

ckpt = torch.load(cfg.ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state"])

ema_shadow = ckpt.get("ema_shadow", None)
if ema_shadow is None:
    raise ValueError("Checkpoint is missing 'ema_shadow'.")

ema.shadow = {k: v.to(device) for k, v in ema_shadow.items()}

y_true, p_pred = predict_val_probs(val_loader, use_ema=True)
metrics_05 = compute_metrics(y_true, p_pred, thr=0.5)

fpr, tpr, thresholds = roc_curve(y_true, p_pred)
j = tpr - fpr
best_idx = int(np.argmax(j))
best_thr = float(thresholds[best_idx]) if np.isfinite(thresholds[best_idx]) else 0.5
metrics_best = compute_metrics(y_true, p_pred, thr=best_thr)

print({"threshold_0.5": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in metrics_05.items()}})
print({"threshold_best": best_thr, "metrics": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in metrics_best.items()}})

plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title(f"ROC Curve (Val)  AUC={metrics_05['auc']:.4f}")
plt.show()

def _plot_confusion(cm: np.ndarray, title: str) -> None:
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.xticks([0, 1], ["Pred 0", "Pred 1"])
    plt.yticks([0, 1], ["True 0", "True 1"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    plt.xlabel("Prediction")
    plt.ylabel("Ground truth")
    plt.show()

cm_05 = confusion_matrix(y_true.astype(np.int32), (p_pred >= 0.5).astype(np.int32), labels=[0, 1])
cm_best = confusion_matrix(y_true.astype(np.int32), (p_pred >= best_thr).astype(np.int32), labels=[0, 1])

_plot_confusion(cm_05, "Confusion Matrix (thr=0.5)")
_plot_confusion(cm_best, f"Confusion Matrix (thr={best_thr:.4f})")

plt.figure()
plt.hist(p_pred[y_true == 0], bins=30, alpha=0.7, label="y=0")
plt.hist(p_pred[y_true == 1], bins=30, alpha=0.7, label="y=1")
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("Validation predicted probability distributions")
plt.legend()
plt.show()

hist_df = pd.DataFrame(history) if "history" in globals() else None
if hist_df is not None and len(hist_df) > 0:
    display(hist_df)

Cell 10 — Explainability (saliency map on the most “important” slice)

In [ ]:
from typing import Dict, Tuple

def _minmax01(a: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    mn = float(np.min(a))
    mx = float(np.max(a))
    return (a - mn) / (mx - mn + eps)

def _load_cached(pid_row: pd.Series) -> Tuple[torch.Tensor, int, str]:
    p = str(pid_row["cache_path"])
    d = np.load(p)
    x = d["x"].astype(np.float32, copy=False)
    y = int(d["y"])
    pid = str(pid_row["BraTS21ID"])
    xt = torch.from_numpy(x).unsqueeze(0).to(device)
    return xt, y, pid

@torch.no_grad()
def _predict_prob(x: torch.Tensor, use_ema: bool = True) -> float:
    model.eval()
    backup = None
    if use_ema:
        backup = ema.apply_to(model)
    z = model(x).detach()
    p = float(torch.sigmoid(z).item())
    if use_ema and backup is not None:
        ema.restore(model, backup)
    return p

def saliency_for_one_patient(pid_row: pd.Series) -> Dict[str, object]:
    x, y, pid = _load_cached(pid_row)
    p = _predict_prob(x, use_ema=True)

    model.eval()
    backup = ema.apply_to(model)

    xg = x.detach().clone().requires_grad_(True)
    z = model(xg).squeeze()
    if z.ndim != 0:
        z = z.mean()
    z.backward()
    grad = xg.grad

    ema.restore(model, backup)

    sal = grad.abs().mean(dim=1).squeeze(0)
    slice_scores = sal.sum(dim=(1, 2))
    k = int(torch.argmax(slice_scores).item())

    x_np = x.squeeze(0).detach().float().cpu().numpy()
    sal_np = sal.detach().float().cpu().numpy()

    return {"pid": pid, "y": y, "p": p, "slice_idx": k, "x": x_np, "sal": sal_np}

ex_row = val_df.sample(n=1, random_state=cfg.seed).iloc[0]
out = saliency_for_one_patient(ex_row)

pid = out["pid"]
y = out["y"]
p = out["p"]
k = out["slice_idx"]
x_np = out["x"]
sal_np = out["sal"]

print({"patient": pid, "true_y": y, "pred_prob": round(p, 6), "slice_idx": k})

order = [("T1w", 0), ("T1wCE", 1), ("T2w", 2), ("FLAIR", 3)]
sal_slice = _minmax01(sal_np[k])

plt.figure(figsize=(10, 10))
for i, (name, ch) in enumerate(order, start=1):
    img = _minmax01(x_np[ch, k])
    plt.subplot(4, 2, 2 * i - 1)
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.title(f"{name} (slice {k})")

    plt.subplot(4, 2, 2 * i)
    plt.imshow(img, cmap="gray")
    plt.imshow(sal_slice, alpha=0.45)
    plt.axis("off")
    plt.title(f"{name} + saliency")

plt.tight_layout()
plt.show()

Cell 11 — K-Fold Cross-Validation (the “research upgrade” to make your AUC trustworthy)

In [ ]:
from sklearn.model_selection import StratifiedKFold

def _make_model_and_ema() -> tuple[nn.Module, EMA]:
    m = MGMTResNet3D(in_ch=4, base=24).to(device)
    e = EMA(m, decay=cfg.ema_decay)
    return m, e

@torch.no_grad()
def _predict_probs(model_: nn.Module, ema_: EMA, loader: DataLoader, use_ema: bool = True) -> tuple[np.ndarray, np.ndarray]:
    model_.eval()
    backup = None
    if use_ema:
        backup = ema_.apply_to(model_)
    ys: list[float] = []
    ps: list[float] = []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        z = model_(x)
        p = torch.sigmoid(z)
        ys.extend(y.detach().float().cpu().numpy().tolist())
        ps.extend(p.detach().float().cpu().numpy().tolist())
    if use_ema and backup is not None:
        ema_.restore(model_, backup)
    return np.asarray(ys, dtype=np.float32), np.asarray(ps, dtype=np.float32)

def _fold_pos_weight(y_train: np.ndarray) -> torch.Tensor:
    pos = int(np.sum(y_train == 1))
    neg = int(np.sum(y_train == 0))
    w = float(neg / max(1, pos))
    return torch.tensor([w], dtype=torch.float32)

def _loader_from_frame(frame: pd.DataFrame, shuffle: bool) -> DataLoader:
    ds = MGMTCachedDataset(frame.reset_index(drop=True), expected_shape=(4, *cfg.target_shape))
    pin = bool(cfg.pin_memory and device.type == "cuda")
    return DataLoader(
        ds,
        batch_size=int(cfg.batch_size),
        shuffle=shuffle,
        num_workers=int(cfg.num_workers),
        pin_memory=pin,
        drop_last=False,
    )

def _train_one_fold(train_frame: pd.DataFrame, val_frame: pd.DataFrame, fold_idx: int, epochs: int) -> dict:
    model_f, ema_f = _make_model_and_ema()

    y_train = train_frame["y"].astype(int).to_numpy()
    pos_weight_f = _fold_pos_weight(y_train).to(device)

    criterion_f = nn.BCEWithLogitsLoss(pos_weight=pos_weight_f)
    optimizer_f = torch.optim.AdamW(model_f.parameters(), lr=float(cfg.lr), weight_decay=float(cfg.weight_decay))
    scheduler_f = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_f, T_max=int(epochs))

    use_amp_f = (device.type == "cuda")
    scaler_f = torch.cuda.amp.GradScaler(enabled=use_amp_f)

    train_loader_f = _loader_from_frame(train_frame, shuffle=True)
    val_loader_f = _loader_from_frame(val_frame, shuffle=False)

    best_auc = -1.0
    best_epoch = 0

    for epoch in range(1, int(epochs) + 1):
        model_f.train()
        running = 0.0
        steps = 0

        for x, y in train_loader_f:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer_f.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp_f):
                z = model_f(x)
                loss = criterion_f(z, y)

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss in fold {fold_idx} epoch {epoch}: {loss.item()}")

            scaler_f.scale(loss).backward()
            scaler_f.unscale_(optimizer_f)
            torch.nn.utils.clip_grad_norm_(model_f.parameters(), float(cfg.grad_clip_norm))
            scaler_f.step(optimizer_f)
            scaler_f.update()

            ema_f.update(model_f)

            running += float(loss.detach().cpu().item())
            steps += 1

        scheduler_f.step()

        yv, pv = _predict_probs(model_f, ema_f, val_loader_f, use_ema=True)
        auc = float(roc_auc_score(yv, pv)) if len(np.unique(yv.astype(int))) > 1 else float("nan")

        if math.isfinite(auc) and auc > best_auc:
            best_auc = auc
            best_epoch = epoch

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "fold": int(fold_idx),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "train_n": int(len(train_frame)),
        "val_n": int(len(val_frame)),
    }

if "cache_path" not in df.columns:
    raise ValueError("df must contain 'cache_path' before running k-fold CV.")

K = 5
CV_EPOCHS = max(2, int(cfg.epochs))

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=int(cfg.seed))
y_all = df["y"].astype(int).to_numpy()

fold_results: list[dict] = []
for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(df, y_all), start=1):
    tr = df.iloc[tr_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    r = _train_one_fold(tr, va, fold_idx=fold_idx, epochs=CV_EPOCHS)
    fold_results.append(r)
    print({"fold": r["fold"], "best_auc": round(r["best_auc"], 6), "best_epoch": r["best_epoch"], "train_n": r["train_n"], "val_n": r["val_n"]})

res_df = pd.DataFrame(fold_results)
auc_vals = res_df["best_auc"].to_numpy(dtype=np.float64)
mean_auc = float(np.mean(auc_vals))
std_auc = float(np.std(auc_vals, ddof=1)) if len(auc_vals) > 1 else 0.0

print({"k": K, "epochs_per_fold": CV_EPOCHS, "mean_auc": round(mean_auc, 6), "std_auc": round(std_auc, 6)})
display(res_df)

Cell 12 — Strong 3D augmentations (train only) without breaking your caching pipeline

In [ ]:
import math
from typing import Optional, Tuple

class Random3DAugment:
    def __init__(
        self,
        p_flip: float = 0.5,
        p_affine: float = 0.7,
        p_intensity: float = 0.8,
        p_noise: float = 0.5,
        rot_deg: float = 10.0,
        scale_range: Tuple[float, float] = (0.9, 1.1),
        trans_frac: float = 0.06,
        intensity_scale_range: Tuple[float, float] = (0.85, 1.15),
        intensity_shift_range: Tuple[float, float] = (-0.15, 0.15),
        noise_std_range: Tuple[float, float] = (0.0, 0.08),
        seed: Optional[int] = None,
    ):
        self.p_flip = float(p_flip)
        self.p_affine = float(p_affine)
        self.p_intensity = float(p_intensity)
        self.p_noise = float(p_noise)
        self.rot_deg = float(rot_deg)
        self.scale_range = (float(scale_range[0]), float(scale_range[1]))
        self.trans_frac = float(trans_frac)
        self.intensity_scale_range = (float(intensity_scale_range[0]), float(intensity_scale_range[1]))
        self.intensity_shift_range = (float(intensity_shift_range[0]), float(intensity_shift_range[1]))
        self.noise_std_range = (float(noise_std_range[0]), float(noise_std_range[1]))
        self.rng = np.random.default_rng(seed)

    def _rand(self) -> float:
        return float(self.rng.random())

    def _uniform(self, a: float, b: float) -> float:
        return float(self.rng.uniform(a, b))

    def _flip(self, x: torch.Tensor) -> torch.Tensor:
        if self._rand() >= self.p_flip:
            return x
        for dim in (2, 3, 4):
            if self._rand() < 0.5:
                x = torch.flip(x, dims=(dim,))
        return x

    def _affine_theta(self, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
        rx = math.radians(self._uniform(-self.rot_deg, self.rot_deg))
        ry = math.radians(self._uniform(-self.rot_deg, self.rot_deg))
        rz = math.radians(self._uniform(-self.rot_deg, self.rot_deg))

        sx = self._uniform(*self.scale_range)
        sy = self._uniform(*self.scale_range)
        sz = self._uniform(*self.scale_range)

        tx = self._uniform(-self.trans_frac, self.trans_frac)
        ty = self._uniform(-self.trans_frac, self.trans_frac)
        tz = self._uniform(-self.trans_frac, self.trans_frac)

        cx, sx_sin = math.cos(rx), math.sin(rx)
        cy, sy_sin = math.cos(ry), math.sin(ry)
        cz, sz_sin = math.cos(rz), math.sin(rz)

        Rx = np.array([[1.0, 0.0, 0.0],
                       [0.0, cx, -sx_sin],
                       [0.0, sx_sin, cx]], dtype=np.float32)
        Ry = np.array([[cy, 0.0, sy_sin],
                       [0.0, 1.0, 0.0],
                       [-sy_sin, 0.0, cy]], dtype=np.float32)
        Rz = np.array([[cz, -sz_sin, 0.0],
                       [sz_sin, cz, 0.0],
                       [0.0, 0.0, 1.0]], dtype=np.float32)

        R = (Rz @ Ry @ Rx).astype(np.float32)
        S = np.diag([1.0 / sx, 1.0 / sy, 1.0 / sz]).astype(np.float32)
        A = (R @ S).astype(np.float32)

        theta = np.zeros((1, 3, 4), dtype=np.float32)
        theta[0, :, :3] = A
        theta[0, :, 3] = np.array([tx, ty, tz], dtype=np.float32)
        return torch.from_numpy(theta).to(device=device, dtype=dtype)

    def _affine(self, x: torch.Tensor) -> torch.Tensor:
        if self._rand() >= self.p_affine:
            return x
        theta = self._affine_theta(dtype=x.dtype, device=x.device)
        grid = F.affine_grid(theta, size=x.size(), align_corners=False)
        return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)

    def _intensity(self, x: torch.Tensor) -> torch.Tensor:
        if self._rand() >= self.p_intensity:
            return x
        a = self._uniform(*self.intensity_scale_range)
        b = self._uniform(*self.intensity_shift_range)
        return x * a + b

    def _noise(self, x: torch.Tensor) -> torch.Tensor:
        if self._rand() >= self.p_noise:
            return x
        s = self._uniform(*self.noise_std_range)
        if s <= 0.0:
            return x
        n = torch.randn_like(x) * s
        return x + n

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        x = self._flip(x)
        x = self._affine(x)
        x = self._intensity(x)
        x = self._noise(x)
        return x


class MGMTCachedAugDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, expected_shape: Tuple[int, int, int, int], augment: Optional[Random3DAugment] = None):
        self.paths = frame["cache_path"].astype(str).tolist()
        self.labels = frame["y"].astype(int).tolist()
        self.expected_shape = expected_shape
        self.augment = augment

        bad = [p for p in self.paths if not Path(p).exists()]
        if bad:
            raise FileNotFoundError(f"Missing cache files (showing up to 3): {bad[:3]}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        p = self.paths[idx]
        d = np.load(p)
        x = d["x"]
        y = float(self.labels[idx])

        if x.shape != self.expected_shape:
            raise ValueError(f"Bad x shape in {p}: {x.shape}, expected {self.expected_shape}")
        if x.dtype != np.float32:
            x = x.astype(np.float32, copy=False)

        xt = torch.from_numpy(x).float()
        if self.augment is not None:
            xt = self.augment(xt.unsqueeze(0)).squeeze(0)

        yt = torch.tensor(y, dtype=torch.float32)
        return xt, yt


augment = Random3DAugment(seed=cfg.seed)

train_aug_set = MGMTCachedAugDataset(train_df, expected_shape=(4, *cfg.target_shape), augment=augment)
val_noaug_set = MGMTCachedAugDataset(val_df, expected_shape=(4, *cfg.target_shape), augment=None)

pin = bool(cfg.pin_memory and device.type == "cuda")

train_loader_aug = DataLoader(
    train_aug_set,
    batch_size=int(cfg.batch_size),
    shuffle=True,
    num_workers=int(cfg.num_workers),
    pin_memory=pin,
    drop_last=False,
)

val_loader_noaug = DataLoader(
    val_noaug_set,
    batch_size=int(cfg.batch_size),
    shuffle=False,
    num_workers=int(cfg.num_workers),
    pin_memory=pin,
    drop_last=False,
)

x1, y1 = next(iter(train_loader_aug))
print({"train_aug_batch_x": tuple(x1.shape), "train_aug_batch_y": tuple(y1.shape), "x_min": float(x1.min()), "x_max": float(x1.max())})

mid = x1.shape[2] // 2
plt.figure(figsize=(10, 3))
plt.subplot(1, 4, 1); plt.imshow(x1[0, 0, mid].cpu().numpy(), cmap="gray"); plt.axis("off"); plt.title("T1w (aug)")
plt.subplot(1, 4, 2); plt.imshow(x1[0, 1, mid].cpu().numpy(), cmap="gray"); plt.axis("off"); plt.title("T1wCE (aug)")
plt.subplot(1, 4, 3); plt.imshow(x1[0, 2, mid].cpu().numpy(), cmap="gray"); plt.axis("off"); plt.title("T2w (aug)")
plt.subplot(1, 4, 4); plt.imshow(x1[0, 3, mid].cpu().numpy(), cmap="gray"); plt.axis("off"); plt.title("FLAIR (aug)")
plt.tight_layout()
plt.show()

Cell 13 — Voxel-spacing resampling (the biggest preprocessing upgrade after augmentation)

In [ ]:
from typing import Sequence, Dict, List, Tuple

TARGET_SPACING = (1.0, 1.0, 1.0)
cache_root_rs = (data_root / f"cache_128_sp_{TARGET_SPACING[0]:.2f}_{TARGET_SPACING[1]:.2f}_{TARGET_SPACING[2]:.2f}").resolve()
cache_root_rs.mkdir(parents=True, exist_ok=True)

def _select_series_id(series_dir: str, series_ids: Sequence[str]) -> str:
    reader = sitk.ImageSeriesReader()
    best_sid = series_ids[0]
    best_key = (-1, -1, -1)

    for sid in series_ids:
        files = reader.GetGDCMSeriesFileNames(series_dir, sid)
        if not files:
            continue
        r = sitk.ImageFileReader()
        r.SetFileName(files[0])
        r.ReadImageInformation()
        sx, sy = r.GetSize()[0], r.GetSize()[1]
        key = (len(files), int(sx), int(sy))
        if key > best_key:
            best_key = key
            best_sid = sid

    if best_key[0] < 1:
        raise FileNotFoundError(f"No readable DICOM slices in: {series_dir}")

    return best_sid

def load_dicom_image(series_dir: str) -> sitk.Image:
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(series_dir)
    if not series_ids:
        raise FileNotFoundError(f"No DICOM series found in: {series_dir}")
    sid = _select_series_id(series_dir, series_ids)
    files = reader.GetGDCMSeriesFileNames(series_dir, sid)
    if len(files) < 8:
        raise ValueError(f"Too few DICOM slices (n={len(files)}) in: {series_dir}")
    reader.SetFileNames(files)
    img = reader.Execute()
    if img.GetDimension() != 3:
        raise ValueError(f"Expected 3D image, got dim={img.GetDimension()} from {series_dir}")
    return sitk.Cast(img, sitk.sitkFloat32)

def resample_to_spacing(img: sitk.Image, spacing: Tuple[float, float, float]) -> sitk.Image:
    orig_spacing = img.GetSpacing()
    orig_size = img.GetSize()

    new_size = [
        int(max(1, round(orig_size[i] * (orig_spacing[i] / spacing[i]))))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0.0)
    out = resampler.Execute(img)

    if out.GetSize()[0] < 2 or out.GetSize()[1] < 2 or out.GetSize()[2] < 2:
        raise ValueError(f"Bad resampled size: {out.GetSize()}")

    return out

def image_to_numpy_dhw(img: sitk.Image) -> np.ndarray:
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D numpy array, got shape {arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError("Non-finite values in converted numpy volume")
    return arr

def zscore_nonzero(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    m = x > 0
    if int(m.sum()) < 64:
        return x.astype(np.float32, copy=False)
    mu = float(x[m].mean())
    sd = float(x[m].std())
    return ((x - mu) / (sd + eps)).astype(np.float32, copy=False)

def center_crop_pad_3d(x: np.ndarray, target: Tuple[int, int, int]) -> np.ndarray:
    td, th, tw = target
    d, h, w = x.shape
    out = np.zeros((td, th, tw), dtype=x.dtype)

    sd0 = max(0, (d - td) // 2)
    sh0 = max(0, (h - th) // 2)
    sw0 = max(0, (w - tw) // 2)

    sd1 = min(d, sd0 + td)
    sh1 = min(h, sh0 + th)
    sw1 = min(w, sw0 + tw)

    dd0 = max(0, (td - d) // 2)
    dh0 = max(0, (th - h) // 2)
    dw0 = max(0, (tw - w) // 2)

    dd1 = dd0 + (sd1 - sd0)
    dh1 = dh0 + (sh1 - sh0)
    dw1 = dw0 + (sw1 - sw0)

    out[dd0:dd1, dh0:dh1, dw0:dw1] = x[sd0:sd1, sh0:sh1, sw0:sw1]
    return out

def preprocess_patient_resampled(row: pd.Series, target_shape: Tuple[int, int, int], target_spacing: Tuple[float, float, float]) -> np.ndarray:
    order = ["T1w", "T1wCE", "T2w", "FLAIR"]
    vols: List[np.ndarray] = []
    for m in order:
        img = load_dicom_image(str(row[m]))
        img = resample_to_spacing(img, target_spacing)
        v = image_to_numpy_dhw(img)
        v = zscore_nonzero(v)
        v = center_crop_pad_3d(v, target_shape)
        if not np.isfinite(v).all():
            raise ValueError(f"Non-finite after preprocessing in {m} for {row['BraTS21ID']}")
        vols.append(v.astype(np.float32, copy=False))
    x = np.stack(vols, axis=0)
    if x.shape != (4, *target_shape):
        raise ValueError(f"Bad stacked shape {x.shape}")
    return x

def cache_path_for_rs(pid: str, target_shape: Tuple[int, int, int], target_spacing: Tuple[float, float, float]) -> Path:
    d, h, w = target_shape
    sx, sy, sz = target_spacing
    return cache_root_rs / f"{pid}_{d}x{h}x{w}_sp_{sx:.2f}_{sy:.2f}_{sz:.2f}.npz"

def write_npz_atomic(path: Path, x: np.ndarray, y: int) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    np.savez_compressed(tmp, x=x.astype(np.float32, copy=False), y=np.int64(int(y)))
    os.replace(tmp, path)

def cache_one_resampled(row: pd.Series) -> str:
    pid = str(row["BraTS21ID"])
    y = int(row["y"])
    out = cache_path_for_rs(pid, cfg.target_shape, TARGET_SPACING)
    if out.exists():
        return str(out)
    x = preprocess_patient_resampled(row, cfg.target_shape, TARGET_SPACING)
    write_npz_atomic(out, x, y)
    return str(out)

cache_paths_rs: List[str] = []
failed_rs: List[str] = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="caching_resampled"):
    try:
        cache_paths_rs.append(cache_one_resampled(row))
    except Exception as e:
        failed_rs.append(f"{row['BraTS21ID']}: {type(e).__name__}: {e}")
        cache_paths_rs.append("")

df_rs = df.copy()
df_rs["cache_path_rs"] = cache_paths_rs
df_rs = df_rs[df_rs["cache_path_rs"].astype(str).str.len() > 0].reset_index(drop=True)

print({"cache_root_rs": str(cache_root_rs), "cached_patients_rs": int(len(df_rs)), "failed_rs": int(len(failed_rs))})
if failed_rs:
    print("First 5 failures:")
    for s in failed_rs[:5]:
        print(s)

t = np.load(df_rs.loc[0, "cache_path_rs"])
print({"example_rs_path": df_rs.loc[0, "cache_path_rs"], "x_shape": tuple(t["x"].shape), "y": int(t["y"])})

Cell 14 — Switch your training pipeline to the resampled cache (df_rs)

In [ ]:
from typing import Tuple

if "df_rs" not in globals():
    raise RuntimeError("df_rs not found. Run Cell 13 first.")

if "cache_path_rs" not in df_rs.columns:
    raise RuntimeError("df_rs must contain 'cache_path_rs'. Run Cell 13 first.")

df_rs = df_rs.copy()
df_rs["y"] = df_rs["y"].astype(int)

if len(df_rs) < 50:
    raise RuntimeError(f"Too few resampled cached patients: {len(df_rs)}")

train_df_rs, val_df_rs = train_test_split(
    df_rs,
    test_size=float(cfg.val_size),
    random_state=int(cfg.seed),
    stratify=df_rs["y"],
)

train_df_rs = train_df_rs.reset_index(drop=True)
val_df_rs = val_df_rs.reset_index(drop=True)

train_pos = int(train_df_rs["y"].sum())
train_neg = int(len(train_df_rs) - train_pos)
pos_weight_value = float(train_neg / max(1, train_pos))
pos_weight_rs = torch.tensor([pos_weight_value], dtype=torch.float32)

print(
    {
        "train_n": int(len(train_df_rs)),
        "val_n": int(len(val_df_rs)),
        "train_pos": train_pos,
        "train_neg": train_neg,
        "pos_weight": pos_weight_value,
    }
)

class MGMTCachedDatasetRS(Dataset):
    def __init__(self, frame: pd.DataFrame, expected_shape: Tuple[int, int, int, int]):
        self.paths = frame["cache_path_rs"].astype(str).tolist()
        self.labels = frame["y"].astype(int).tolist()
        self.expected_shape = expected_shape

        bad = [p for p in self.paths if not Path(p).exists()]
        if bad:
            raise FileNotFoundError(f"Missing resampled cache files (showing up to 3): {bad[:3]}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        p = self.paths[idx]
        d = np.load(p)
        x = d["x"]
        y = float(self.labels[idx])

        if x.shape != self.expected_shape:
            raise ValueError(f"Bad x shape in {p}: {x.shape}, expected {self.expected_shape}")
        if x.dtype != np.float32:
            x = x.astype(np.float32, copy=False)

        xt = torch.from_numpy(x).float()
        yt = torch.tensor(y, dtype=torch.float32)
        return xt, yt

expected = (4, *cfg.target_shape)
train_set_rs = MGMTCachedDatasetRS(train_df_rs, expected_shape=expected)
val_set_rs = MGMTCachedDatasetRS(val_df_rs, expected_shape=expected)

pin = bool(cfg.pin_memory and device.type == "cuda")

train_loader_rs = DataLoader(
    train_set_rs,
    batch_size=int(cfg.batch_size),
    shuffle=True,
    num_workers=int(cfg.num_workers),
    pin_memory=pin,
    drop_last=False,
)

val_loader_rs = DataLoader(
    val_set_rs,
    batch_size=int(cfg.batch_size),
    shuffle=False,
    num_workers=int(cfg.num_workers),
    pin_memory=pin,
    drop_last=False,
)

xb, yb = next(iter(train_loader_rs))
print({"batch_x": tuple(xb.shape), "batch_y": tuple(yb.shape), "x_dtype": str(xb.dtype), "y_dtype": str(yb.dtype)})

train_df = train_df_rs
val_df = val_df_rs
train_loader = train_loader_rs
val_loader = val_loader_rs
pos_weight = pos_weight_rs

Cell 15 — Re-train from scratch using your current loaders (now resampled cache)

In [ ]:
import math
from pathlib import Path
from typing import Dict, List, Tuple

if "train_loader" not in globals() or "val_loader" not in globals() or "pos_weight" not in globals():
    raise RuntimeError("Missing train_loader/val_loader/pos_weight. Run the previous cells in order.")

if "MGMTResNet3D" not in globals() or "EMA" not in globals():
    raise RuntimeError("Missing model classes. Run the model cell first.")

if "predict_val_probs" not in globals():
    @torch.no_grad()
    def predict_val_probs(loader: DataLoader, use_ema: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        model.eval()
        backup = None
        if use_ema:
            backup = ema.apply_to(model)
        ys: List[float] = []
        ps: List[float] = []
        for x, y in loader:
            x = x.to(device)
            z = model(x)
            p = torch.sigmoid(z)
            ys.extend(y.detach().float().cpu().numpy().tolist())
            ps.extend(p.detach().float().cpu().numpy().tolist())
        if use_ema and backup is not None:
            ema.restore(model, backup)
        return np.asarray(ys, dtype=np.float32), np.asarray(ps, dtype=np.float32)

if "compute_metrics" not in globals():
    def compute_metrics(y_true: np.ndarray, p_pred: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
        y_true_i = y_true.astype(np.int32)
        y_hat = (p_pred >= thr).astype(np.int32)
        cm = confusion_matrix(y_true_i, y_hat, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        acc = (tp + tn) / max(1, (tp + tn + fp + fn))
        tpr = tp / max(1, (tp + fn))
        tnr = tn / max(1, (tn + fp))
        auc = float("nan")
        if len(np.unique(y_true_i)) > 1:
            auc = float(roc_auc_score(y_true, p_pred))
        return {"auc": float(auc), "acc": float(acc), "tpr": float(tpr), "tnr": float(tnr), "tp": float(tp), "tn": float(tn), "fp": float(fp), "fn": float(fn)}

ckpt_path = Path("best_resampled.pt")

model = MGMTResNet3D(in_ch=4, base=24).to(device)
ema = EMA(model, decay=cfg.ema_decay)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg.lr), weight_decay=float(cfg.weight_decay))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(cfg.epochs))

use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_auc = -1.0
history_resampled: List[Dict[str, float]] = []

for epoch in range(1, int(cfg.epochs) + 1):
    model.train()
    running_loss = 0.0
    steps = 0

    for x, y in tqdm(train_loader, desc=f"train {epoch}/{cfg.epochs}"):
        x = x.to(device, non_blocking=(device.type == "cuda"))
        y = y.to(device, non_blocking=(device.type == "cuda"))

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            z = model(x)
            loss = criterion(z, y)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.grad_clip_norm))
        scaler.step(optimizer)
        scaler.update()

        ema.update(model)

        running_loss += float(loss.detach().cpu().item())
        steps += 1

    scheduler.step()

    train_loss = running_loss / max(1, steps)
    y_true, p_pred = predict_val_probs(val_loader, use_ema=True)
    m = compute_metrics(y_true, p_pred, thr=0.5)

    rec = {"epoch": float(epoch), "train_loss": float(train_loss), **m}
    history_resampled.append(rec)

    print(
        {
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_auc": None if not math.isfinite(m["auc"]) else round(m["auc"], 6),
            "val_acc": round(m["acc"], 6),
            "val_tpr": round(m["tpr"], 6),
            "val_tnr": round(m["tnr"], 6),
            "tp": int(m["tp"]),
            "tn": int(m["tn"]),
            "fp": int(m["fp"]),
            "fn": int(m["fn"]),
        }
    )

    if math.isfinite(m["auc"]) and m["auc"] > best_auc:
        best_auc = float(m["auc"])
        payload = {
            "epoch": int(epoch),
            "best_auc": float(best_auc),
            "cfg": cfg.__dict__,
            "model_state": model.state_dict(),
            "ema_shadow": {k: v.detach().cpu() for k, v in ema.shadow.items()},
        }
        torch.save(payload, ckpt_path)

print({"best_auc": float(best_auc), "checkpoint": str(ckpt_path.resolve())})


Cell 16 — Final evaluation (loads best checkpoint + ROC curve + confusion matrices)

In [ ]:
import math
from pathlib import Path

ckpt_path = Path("best_resampled.pt")
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path.resolve()} (Run Cell 15 first)")

ckpt = torch.load(ckpt_path, map_location="cpu")

model.load_state_dict(ckpt["model_state"])
ema.shadow = {k: v.to(device) for k, v in ckpt["ema_shadow"].items()}

y_true, p_pred = predict_val_probs(val_loader, use_ema=True)

auc = float("nan")
if len(np.unique(y_true.astype(int))) > 1:
    auc = float(roc_auc_score(y_true, p_pred))

fpr, tpr, thresholds = roc_curve(y_true, p_pred)
j = tpr - fpr
best_idx = int(np.argmax(j))
best_thr = float(thresholds[best_idx]) if thresholds.size > 0 and np.isfinite(thresholds[best_idx]) else 0.5

m_05 = compute_metrics(y_true, p_pred, thr=0.5)
m_best = compute_metrics(y_true, p_pred, thr=best_thr)

print(
    {
        "val_auc": None if not math.isfinite(auc) else round(auc, 6),
        "thr_0.5": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in m_05.items()},
        "best_thr": round(best_thr, 6),
        "thr_best": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in m_best.items()},
    }
)

plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title(f"ROC Curve (Val)  AUC={auc:.4f}")
plt.show()

cm_05 = confusion_matrix(y_true.astype(np.int32), (p_pred >= 0.5).astype(np.int32), labels=[0, 1])
cm_best = confusion_matrix(y_true.astype(np.int32), (p_pred >= best_thr).astype(np.int32), labels=[0, 1])

def plot_cm(cm: np.ndarray, title: str) -> None:
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.xticks([0, 1], ["Pred 0", "Pred 1"])
    plt.yticks([0, 1], ["True 0", "True 1"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    plt.xlabel("Prediction")
    plt.ylabel("Ground truth")
    plt.show()

plot_cm(cm_05, "Confusion Matrix (thr=0.5)")
plot_cm(cm_best, f"Confusion Matrix (thr={best_thr:.4f})")

plt.figure()
plt.hist(p_pred[y_true == 0], bins=30, alpha=0.7, label="y=0")
plt.hist(p_pred[y_true == 1], bins=30, alpha=0.7, label="y=1")
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("Validation predicted probability distributions")
plt.legend()
plt.show()

if "history_resampled" in globals():
    display(pd.DataFrame(history_resampled))